# Three shared worry-only clusters, 2019–2025

This is separate from the main five-cluster worry–harm analysis. It uses the five worry questions available in all four waves, with one independently fitted model per wave. Run All checks the saved model centres, all confirmation assignments, the exact assignment rule, and the separate short descriptions. Inputs contain aggregated answer-pattern counts and weights, plus fitted centres and model outputs for those patterns. No model fitting is started.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score
from IPython.display import display

assets=next(folder/'assets' for cwd in [Path.cwd(),*Path.cwd().parents]
            for folder in [cwd,cwd/'paper',cwd/'github_publication/paper']
            if (folder/'assets/shared_worry_only_manifest.json').is_file())
manifest=json.loads((assets/'shared_worry_only_manifest.json').read_text())
for path,h in manifest['files_sha256'].items():
    assert hashlib.sha256((assets/path).read_bytes()).hexdigest()==h
report=json.loads((assets/'shared_worry_only_report.json').read_text())
conditions=json.loads((assets/'shared_worry_only_conditions.json').read_text())
with np.load(assets/'shared_worry_only_evidence.npz',allow_pickle=False) as z:
    x,counts,mass,native,labels,centres=(z[k] for k in [
        'answers','reference_counts','reference_mass','native_mass','fit_labels','centres'])
assert len(x)==3**5 and labels.shape==(4,101,243)
assert counts.sum()==91249

## Exact assignment

The transformed five worry values use 0, 0.75 and 1. The model fits the five differences from each respondent's mean, and the mean as a separate coordinate. Its squared distance coefficient is 1.5 times that for each centred coordinate. All five answers enter the clustering.

To assign a complete response, check crime first. If answered *not worried*, assign the no-crime-worry cluster. Otherwise check water. If answered *not worried*, assign crime worry without water worry. Otherwise assign crime and water worry. This rule is disjoint and gives every complete response one cluster.

In [2]:
v=np.where(x==2,1.,np.where(x==1,.75,0.))
m=v.mean(1,keepdims=True)
z=np.column_stack([v-m,np.sqrt(1.5)*m])/np.sqrt(5)
exact=np.where(x[:,2]==0,1,np.where(x[:,1]==0,0,2))
for c,local in zip(centres.reshape(-1,3,6),labels.reshape(-1,243)):
    np.testing.assert_array_equal(cdist(z,c).argmin(1),local)
    tab=np.bincount(3*exact+local,weights=mass,minlength=9).reshape(3,3)
    a,b=linear_sum_assignment(-tab)
    mapping=np.empty(3,dtype=int)
    mapping[b]=a
    np.testing.assert_array_equal(mapping[local],exact)
    assert adjusted_rand_score(np.repeat(mapping[local],counts),np.repeat(exact,counts))==1.
print('All 4 point fits and all 400 refits agree on all 243 complete answer patterns.')
print('Every cross-wave comparison has ARI and weighted Jaccard of 1.')

All 4 point fits and all 400 refits agree on all 243 complete answer patterns.
Every cross-wave comparison has ARI and weighted Jaccard of 1.


## The size limit and the short descriptions

The original maximum cluster share is 40%. The prespecified sensitivity analysis allows 45%. The largest share is 44.63% in the 2019 native development population. This is why the exact four-wave result qualifies under the sensitivity criterion.

These are native development shares, with equal weight per country and relative survey weights within each country. The country sets differ by wave. In the second table, the rule for crime and water worry uses water alone. That is a shorter description that meets the precision floor; the exact assignment above still checks crime first.

In [3]:
shares=np.column_stack([native[:,exact==c].sum(1) for c in range(3)])
np.testing.assert_allclose(shares,report['native_shares'],atol=1e-12)
assert report['original_cap']<shares.max()<=report['sensitivity_cap']
display(pd.DataFrame(100*shares.T,index=report['names'],columns=report['waves']).round(2))
answers={'not':(x==0).sum(1),'very':(x==2).sum(1),'some':(x>=1).sum(1)}
ops={'eq':np.equal,'le':np.less_equal,'ge':np.greater_equal}
tests=np.column_stack([ops[t['op']](x[:,t['j']] if 'j' in t else answers[t['base']],t['value'])
                       for t in conditions])
for rule in report['rules']:
    mask=np.logical_or.reduce([tests[:,part].all(1) for part in rule['formula']['clauses']])
    member=exact==rule['cluster']
    joint=native[:,mask&member].sum(1)
    precision=joint/native[:,mask].sum(1)
    coverage=joint/native[:,member].sum(1)
    np.testing.assert_allclose(precision,rule['precision'],atol=1e-12)
    np.testing.assert_allclose(coverage,rule['coverage'],atol=1e-12)
    assert precision.min()>=.8 and np.allclose(coverage,1.)
display(pd.DataFrame([{'Cluster':r['name'],'Rule':r['text'],'Cost':r['cost'],
                       'Minimum precision (%)':100*min(r['precision']),
                       'Minimum coverage (%)':100*min(r['coverage'])}
                     for r in report['rules']]).round(2))

,2019,2021,2023,2025
Crime worry without water worry,27.55,27.86,29.42,28.20
No crime worry,27.82,34.68,33.45,32.85
Crime and water worry,44.63,37.46,37.13,38.95


,Cluster,Rule,Cost,Minimum precision (%),Minimum coverage (%)
0,Crime worry without water worry,water: not worried AND crime: at least somewha...,2,100.00,100.0
1,No crime worry,crime: not worried,1,100.00,100.0
2,Crime and water worry,water: at least somewhat worried,1,82.02,100.0
